# Problem 2. 뉴스 제목으로 기사 주제 분류하기

## 문제 개요
포털 뉴스 편집팀은 쏟아지는 기사를 **제목만 보고** 7개 섹션으로 자동 분류하려 합니다.
뉴스 제목으로 기사 주제를 예측하세요.

## 파일 구조
```
Problem2/
├── problem2.ipynb      ← 이 노트북
├── submission.csv        ← 제출 파일 (직접 생성)
└── data/
    ├── train.csv
    ├── test.csv
    └── sample_submission.csv
```

## 데이터
- `data/train.csv` : 30,000개 (label 포함)
- `data/test.csv` : 10,000개 (label 없음)

| 컬럼 | 설명 |
|---|---|
| id | 기사 고유 번호 (`N00000` 형식) |
| title | 뉴스 제목 |
| **label** | **주제: IT과학, 경제, 사회, 생활문화, 세계, 스포츠, 정치 (예측 대상)** |

### 데이터 특징
- 제목이 짧고(평균 27자) 한자 약어(美, 中, 北, 與)와 기호가 많습니다.
- 사회·생활문화처럼 경계가 모호한 주제가 있습니다.
- 출처: KLUE-YNAT (KLUE Benchmark, CC BY-SA 4.0) 학습 데이터 일부를 대회용으로 재구성

## 규칙
- 제한 시간: 3문제 합계 **90분**
- 허용: pandas, scikit-learn, LightGBM, XGBoost, OpenCV, PyTorch 등 일반 라이브러리와 공개 사전학습 모델
- 금지: LLM API로 테스트 데이터를 직접 예측, 테스트 데이터 수작업 라벨링, 원본 공개 데이터셋에서 정답 조회
- 결과는 이 노트북과 같은 위치(`Problem2/`)에 **`submission.csv`** 로 저장

## 평가
**Macro F1** — 높을수록 좋습니다. 7개 주제 각각의 F1을 단순 평균합니다.

## 초기 코드 (베이스라인)
아래 베이스라인은 자주 나오는 단어 1,000개만 세어 나이브 베이즈로 분류합니다. 동작은 하지만 점수가 낮습니다.

## 최종 풀이: 글자 n-gram TF-IDF + LinearSVC
- 제목이 짧고(평균 27자) 띄어쓰기·어미 변형이 많아 단어 대신 **글자 1~4-gram**(`char_wb`)을 씁니다.
- 5-Fold OOF Macro F1(실측): 단어 1,000개 NB 베이스라인은 미측정, 글자 NB 0.823, LR 0.832, **LinearSVC 0.841**. NB+SVC 순위 평균 0.829로 오히려 낮았고, C·n-gram 범위·단어 특징·class_weight 변형은 모두 0.834~0.841(노이즈 수준)이라 여기서 멈췄습니다.

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

train = pd.read_csv("data/train.csv"); test = pd.read_csv("data/test.csv")
classes = sorted(train.label.unique()); y = train.label.map({c: i for i, c in enumerate(classes)}).values
T, Tt = train.title.values, test.title.values

def fit_predict(ta, ya, tb):
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 4), sublinear_tf=True, min_df=2)
    return LinearSVC(C=0.3).fit(v.fit_transform(ta), ya).decision_function(v.transform(tb))

oof = np.zeros((len(y), len(classes)))
for a, b in StratifiedKFold(5, shuffle=True, random_state=0).split(T, y): oof[b] = fit_predict(T[a], y[a], T[b])
print("5-Fold OOF Macro F1: %.4f" % f1_score(y, oof.argmax(1), average="macro"))

pred = fit_predict(T, y, Tt).argmax(1)
sub = pd.DataFrame({"id": test.id, "label": np.array(classes)[pred]})
assert list(sub.columns) == ["id", "label"] and len(sub) == len(test) and set(sub.label) <= set(classes)
sub.to_csv("submission.csv", index=False)
print(sub.label.value_counts(normalize=True).round(3).to_dict()); sub.head()

5-Fold OOF Macro F1: 0.8409


{'세계': 0.189, '스포츠': 0.169, '정치': 0.167, '생활문화': 0.134, '경제': 0.13, 'IT과학': 0.126, '사회': 0.085}


,id,label
0,N30000,정치
1,N30001,정치
2,N30002,경제
3,N30003,생활문화
4,N30004,IT과학


## 제출 형식
`submission.csv`

| id | label |
|---|---|
| N30000 | 스포츠 |
| N30001 | 경제 |

- 컬럼명 `id`, `label`, label은 위 7개 주제 이름 그대로
- 포맷이 틀리면 0점 처리됩니다.

In [2]:
import pandas as pd
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()

(10000, 2)


,id,label
0,N30000,정치
1,N30001,정치
2,N30002,경제
3,N30003,생활문화
4,N30004,IT과학
